In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

print("현재 작업 폴더:", Path.cwd())

env_path = Path.cwd() / ".env"

print(".env 파일 존재 여부:", env_path.exists())

if env_path.exists():
    load_dotenv(env_path, override=True)

print("Gemini 키 인식 여부:", bool(os.getenv("GEMINI_API_KEY")))

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from google import genai

# .env 파일 불러오기
load_dotenv(Path.cwd() / ".env", override=True)

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("Gemini API 키를 찾을 수 없습니다.")

# Gemini 클라이언트 생성
client = genai.Client(api_key=api_key)

print("Gemini 클라이언트 생성 완료!")

In [ ]:
import json
from google.genai import types

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="""
    음식명: 김치볶음밥

    이 음식에 일반적으로 사용되는 재료를 추정해줘.

    조건:
    - 일반적인 재료만 포함
    - 선택적 토핑 제외
    - 실제 식당의 재료를 확인한 것은 아님
    - 재료명만 반환
    """,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema={
            "type": "OBJECT",
            "properties": {
                "menu_name": {"type": "STRING"},
                "estimated_ingredients": {
                    "type": "ARRAY",
                    "items": {"type": "STRING"}
                },
                "source": {"type": "STRING"}
            },
            "required": [
                "menu_name",
                "estimated_ingredients",
                "source"
            ]
        }
    )
)

result = json.loads(response.text)

print(result)
print("추정 재료:", result["estimated_ingredients"])

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path.cwd()

# 공공데이터 매핑 결과
matching = pd.read_csv(
    BASE / "menu_matching_classified.csv",
    dtype={"preprocess_row_id": str},
    low_memory=False
)

# Qwen 재료 추출 결과
qwen = pd.read_csv(
    BASE / "local_llm_output_v5_full" /
    "llm_ingredient_extraction_v5.csv",
    dtype={"preprocess_row_id": str},
    low_memory=False
)

# 공공데이터 재료 정보
public = pd.read_csv(
    BASE / "public_ingredients_clean.csv",
    low_memory=False
)

print("공공데이터 매핑:", len(matching), "개")
print("Qwen 분석:", len(qwen), "개")
print("공공데이터 재료:", len(public), "행")

print("\n[매핑 데이터 컬럼]")
print(matching.columns.tolist())

print("\n[Qwen 데이터 컬럼]")
print(qwen.columns.tolist())

print("\n[공공데이터 재료 컬럼]")
print(public.columns.tolist())

In [ ]:
columns = [
    "preprocess_row_id",
    "explicit_ingredients",
    "explicit_components",
    "evidence_level",
    "information_status",
    "extraction_status"
]

available = [c for c in columns if c in qwen.columns]

display(qwen[available].head(10))

In [ ]:
import json
import pandas as pd

# Qwen 재료 목록을 실제 Python 리스트로 변환
def parse_list(value):
    if isinstance(value, list):
        return value
    if pd.isna(value) or str(value).strip() == "":
        return []
    try:
        result = json.loads(value)
        return result if isinstance(result, list) else []
    except (ValueError, TypeError):
        return []

# 필요한 Qwen 컬럼만 선택
qwen_selected = qwen[
    [
        "preprocess_row_id",
        "explicit_ingredients",
        "explicit_components",
        "evidence_level",
        "information_status"
    ]
].copy()

# 중복 ID 확인
assert matching["preprocess_row_id"].is_unique
assert qwen_selected["preprocess_row_id"].is_unique

# 매핑 데이터 + Qwen 결과 연결
merged = matching.merge(
    qwen_selected,
    on="preprocess_row_id",
    how="left",
    validate="one_to_one"
)

# 추출된 재료 및 구성 음식 변환
merged["qwen_ingredients"] = merged[
    "explicit_ingredients"
].apply(parse_list)

merged["qwen_components"] = merged[
    "explicit_components"
].apply(parse_list)

# 재료 보유 여부
merged["has_qwen_ingredients"] = merged[
    "qwen_ingredients"
].apply(lambda x: len(x) > 0)

# 구성 음식 보유 여부
merged["has_qwen_components"] = merged[
    "qwen_components"
].apply(lambda x: len(x) > 0)

print("통합 메뉴 수:", len(merged))
print("Qwen 재료 보유:", merged["has_qwen_ingredients"].sum())
print("Qwen 재료 없음:", (~merged["has_qwen_ingredients"]).sum())
print("구성 음식 보유:", merged["has_qwen_components"].sum())

print("\n[매칭 상태별 메뉴 수]")
print(merged["match_review_status"].value_counts(dropna=False))

print("\n[매핑 관련 컬럼]")
print([
    c for c in merged.columns
    if any(k in c.lower() for k in
           ["match", "fd_code", "verified", "approved"])
])

In [ ]:
# 자동 매칭 상태 확인
print("[공공데이터 자동 매칭 상태]")
print(
    merged["match_status_provisional"]
    .value_counts(dropna=False)
)

# Qwen 재료 보유 여부와 비교
print("\n[매칭 상태 × Qwen 재료 보유 여부]")
print(
    pd.crosstab(
        merged["match_status_provisional"],
        merged["has_qwen_ingredients"],
        margins=True
    )
)

# 공공데이터 Top1 음식 코드 존재 여부
merged["has_public_candidate"] = (
    merged["top1_fd_Code"]
    .notna()
    & merged["top1_fd_Code"].astype(str).str.strip().ne("")
)

print("\n[공공데이터 후보 보유 여부]")
print(
    merged["has_public_candidate"]
    .value_counts()
)

# 공공데이터 후보도 없고 Qwen 재료도 없는 메뉴
gemini_priority = merged[
    (~merged["has_public_candidate"])
    & (~merged["has_qwen_ingredients"])
].copy()

print("\nGemini 우선 분석 대상:", len(gemini_priority), "개")

display(
    gemini_priority[
        [
            "preprocess_row_id",
            "restaurant_name",
            "menu_name",
            "description",
            "qwen_components"
        ]
    ].head(10)
)

In [ ]:
# Qwen 재료가 없는 메뉴부터 확인
needs_review = merged[
    ~merged["has_qwen_ingredients"]
].copy()

print("Qwen 재료가 없는 메뉴:", len(needs_review))

print("\n[공공데이터 후보 보유 여부]")
print(needs_review["has_public_candidate"].value_counts())

# 음식명이 너무 짧거나 옵션으로 보이는 항목은 별도 분류
import re

def is_unclear_menu(name):
    name = str(name).strip()

    if not name or name.lower() == "nan":
        return True

    if re.fullmatch(r"\d+\s*인(분)?", name):
        return True

    if name in ["추가", "변경", "선택", "기본"]:
        return True

    return False

needs_review["unclear_menu"] = (
    needs_review["menu_name"].apply(is_unclear_menu)
)

# Gemini 보완 검토 대상
gemini_candidates = needs_review[
    ~needs_review["unclear_menu"]
].copy()

print("\nGemini 보완 검토 대상:", len(gemini_candidates))
print("음식명 불명확:", needs_review["unclear_menu"].sum())

display(
    gemini_candidates[
        [
            "preprocess_row_id",
            "restaurant_name",
            "menu_name",
            "description",
            "qwen_components",
            "top1_fd_Code"
        ]
    ].head(10)
)

In [ ]:
import json
import time
import pandas as pd
from google.genai import types

# 기존 Gemini 후보 중 10개만 테스트
test_menus = gemini_candidates.head(10).copy()

results = []

for _, row in test_menus.iterrows():

    prompt = f"""
너는 음식 재료 분석 AI야.

식당명: {row['restaurant_name']}
메뉴명: {row['menu_name']}
메뉴 설명: {row['description']}
기존에 추출한 구성 음식: {row['qwen_components']}

위 정보를 참고해서 이 메뉴에 일반적으로 들어갈 것으로 예상되는
재료를 추정해줘.

조건:
1. 일반적으로 사용되는 재료만 추정할 것
2. 선택적 토핑과 특이한 재료는 제외할 것
3. 세트 메뉴는 구성 음식 전체를 고려할 것
4. 실제 식당의 원재료를 확인한 것처럼 표현하지 말 것
5. 재료명만 출력할 것
"""

    try:
        response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema={
                    "type": "OBJECT",
                    "properties": {
                        "estimated_ingredients": {
                            "type": "ARRAY",
                            "items": {"type": "STRING"}
                        }
                    },
                    "required": ["estimated_ingredients"]
                }
            )
        )

        data = json.loads(response.text)

        results.append({
            "preprocess_row_id": row["preprocess_row_id"],
            "menu_name": row["menu_name"],
            "gemini_estimated_ingredients": json.dumps(
                data["estimated_ingredients"],
                ensure_ascii=False
            ),
            "source": "LLM_ESTIMATED",
            "status": "success"
        })

        print(f"완료: {row['menu_name']}")

    except Exception as e:
        results.append({
            "preprocess_row_id": row["preprocess_row_id"],
            "menu_name": row["menu_name"],
            "gemini_estimated_ingredients": "[]",
            "source": "LLM_ESTIMATED",
            "status": f"error: {type(e).__name__}"
        })

        print(f"오류: {row['menu_name']} - {type(e).__name__}")

    # 호출 간격 조절
    time.sleep(2)

# 결과 저장
gemini_test = pd.DataFrame(results)

output_path = BASE / "gemini_ingredient_test_10.csv"
gemini_test.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n테스트 완료!")
print("저장 위치:", output_path)

display(gemini_test)

In [ ]:
import json
import time
import random
from google.genai import types, errors

# 기존 테스트에서 실패한 메뉴만 선택
failed_ids = gemini_test.loc[
    gemini_test["status"] != "success",
    "preprocess_row_id"
]

retry_menus = test_menus[
    test_menus["preprocess_row_id"].isin(failed_ids)
]

for _, row in retry_menus.iterrows():

    prompt = f"""
    너는 음식 재료 분석 AI야.

    식당명: {row['restaurant_name']}
    메뉴명: {row['menu_name']}
    메뉴 설명: {row['description']}
    구성 음식: {row['qwen_components']}

    이 메뉴에 일반적으로 사용될 것으로 예상되는 재료를 추정해줘.

    조건:
    - 세트 메뉴는 구성 음식 전체를 고려할 것
    - 선택적 토핑은 제외할 것
    - 실제 식당의 확인된 재료라고 주장하지 말 것
    - 재료명만 반환할 것
    """

    for attempt in range(5):
        try:
            response = client.models.generate_content(
                model="gemini-3.5-flash-lite",
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema={
                        "type": "OBJECT",
                        "properties": {
                            "estimated_ingredients": {
                                "type": "ARRAY",
                                "items": {"type": "STRING"}
                            }
                        },
                        "required": ["estimated_ingredients"]
                    }
                )
            )

            data = json.loads(response.text)

            mask = (
                gemini_test["preprocess_row_id"]
                == row["preprocess_row_id"]
            )

            gemini_test.loc[
                mask, "gemini_estimated_ingredients"
            ] = json.dumps(
                data["estimated_ingredients"],
                ensure_ascii=False
            )

            gemini_test.loc[mask, "status"] = "success"

            print(f"재시도 성공: {row['menu_name']}")
            break

        except errors.ServerError as e:
            if e.code not in (500, 502, 503, 504):
                raise

            if attempt == 4:
                print(f"재시도 실패: {row['menu_name']}")
                break

            wait = min(60, 2 ** (attempt + 1) + random.random())
            print(f"서버 오류. {wait:.1f}초 후 재시도")
            time.sleep(wait)

# 결과 CSV 업데이트
gemini_test.to_csv(
    BASE / "gemini_ingredient_test_10.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n최종 테스트 결과")
print(gemini_test["status"].value_counts())

In [ ]:
# 공공데이터 음식 코드와 재료 데이터 연결 여부 확인

# 코드 형식 통일
merged["top1_fd_Code"] = (
    merged["top1_fd_Code"]
    .astype("string")
    .str.strip()
)

public["fd_Code"] = (
    public["fd_Code"]
    .astype("string")
    .str.strip()
)

# 공공데이터 음식 코드별 행 수
public_counts = (
    public.dropna(subset=["fd_Code"])
    .groupby("fd_Code")
    .size()
)

# 매칭 후보에 해당하는 공공데이터 행 존재 여부
merged["public_ingredient_rows"] = (
    merged["top1_fd_Code"]
    .map(public_counts)
    .fillna(0)
    .astype(int)
)

print("[공공데이터 연결 결과]")
print("전체 메뉴:", len(merged))
print(
    "공공데이터 재료 행 연결:",
    (merged["public_ingredient_rows"] > 0).sum()
)
print(
    "공공데이터 재료 행 미연결:",
    (merged["public_ingredient_rows"] == 0).sum()
)

print("\n[Qwen 재료 없는 메뉴 중 공공데이터 연결 여부]")

print(
    pd.crosstab(
        merged["has_qwen_ingredients"],
        merged["public_ingredient_rows"] > 0,
        margins=True
    )
)

In [ ]:
# 공공데이터 컬럼 전체 확인
print("[공공데이터 컬럼]")
for col in public.columns:
    print(col)

print("\n[공공데이터 샘플 5행]")
display(public.head(5))

print("\n[매칭 결과 샘플]")
display(
    merged[
        [
            "preprocess_row_id",
            "menu_name",
            "top1_fd_Code",
            "public_ingredient_rows",
            "has_qwen_ingredients"
        ]
    ].head(10)
)

In [ ]:
import json
import time
import random
import pandas as pd
from pathlib import Path
from google.genai import types, errors

BASE = Path.cwd()
OUTPUT = BASE / "gemini_ingredient_results.csv"
TEST_FILE = BASE / "gemini_ingredient_test_10.csv"

MODEL = "gemini-3.5-flash-lite"
LIMIT = len(gemini_candidates)

# 기존 결과 불러오기
if OUTPUT.exists():
    results = pd.read_csv(
        OUTPUT, dtype={"preprocess_row_id": str}
    )
elif TEST_FILE.exists():
    results = pd.read_csv(
        TEST_FILE, dtype={"preprocess_row_id": str}
    )
else:
    results = pd.DataFrame(columns=[
        "preprocess_row_id",
        "menu_name",
        "gemini_estimated_ingredients",
        "source",
        "status"
    ])

# 기존 성공 결과는 다시 호출하지 않음
completed = set(
    results.loc[
        results["status"] == "success",
        "preprocess_row_id"
    ]
)

targets = gemini_candidates.head(LIMIT).copy()

print("분석 대상:", len(targets))
print(
    "기존 성공 결과:",
    targets["preprocess_row_id"].isin(completed).sum()
)

for _, row in targets.iterrows():
    menu_id = row["preprocess_row_id"]

    if menu_id in completed:
        continue

    description = (
        str(row["description"])
        if pd.notna(row["description"])
        else "정보 없음"
    )

    prompt = f"""
너는 음식 재료 분석 AI야.

식당명: {row['restaurant_name']}
메뉴명: {row['menu_name']}
메뉴 설명: {description}
기존 추출 구성 음식: {row['qwen_components']}

이 메뉴에 일반적으로 사용될 것으로 예상되는 재료를 추정해줘.

조건:
1. 메뉴 설명과 구성 음식을 우선 참고
2. 일반적인 재료만 추정
3. 선택적 토핑 제외
4. 세트 메뉴는 구성 음식 전체 고려
5. 실제 식당에서 사용하는 재료라고 단정하지 않기
6. 재료명만 배열로 반환
"""

    ingredients = []
    status = "error"

    for attempt in range(5):
        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema={
                        "type": "OBJECT",
                        "properties": {
                            "estimated_ingredients": {
                                "type": "ARRAY",
                                "items": {"type": "STRING"}
                            }
                        },
                        "required": ["estimated_ingredients"]
                    }
                )
            )

            data = json.loads(response.text)
            ingredients = data["estimated_ingredients"]
            status = "success"
            break

        except errors.ServerError as e:
            if e.code not in (500, 502, 503, 504):
                status = f"error_{e.code}"
                break

            if attempt == 4:
                status = f"error_{e.code}"
                break

            wait = min(60, 2 ** (attempt + 1) + random.random())
            print(f"서버 오류, {wait:.1f}초 후 재시도")
            time.sleep(wait)

        except Exception as e:
            status = f"error_{type(e).__name__}"
            print(f"오류 상세: {e}")
            break

    # 기존 실패 기록이 있다면 최신 결과로 교체
    results = results[
        results["preprocess_row_id"] != menu_id
    ]

    new_row = pd.DataFrame([{
        "preprocess_row_id": menu_id,
        "menu_name": row["menu_name"],
        "gemini_estimated_ingredients": json.dumps(
            ingredients, ensure_ascii=False
        ),
        "source": "LLM_ESTIMATED",
        "status": status
    }])

    results = pd.concat(
        [results, new_row], ignore_index=True
    )

    if status == "success":
        completed.add(menu_id)

    # 매 메뉴마다 저장: 중단돼도 결과 보존
    results.to_csv(
        OUTPUT, index=False, encoding="utf-8-sig"
    )

    print(f"{menu_id} | {row['menu_name']} | {status}")

    time.sleep(2)

print("\n분석 종료")
print("저장 파일:", OUTPUT)

target_results = results[
    results["preprocess_row_id"].isin(
        targets["preprocess_row_id"]
    )
]

print(target_results["status"].value_counts())
display(target_results.head(10))

In [ ]:
import pandas as pd
import json

results = pd.read_csv("gemini_ingredient_results.csv")

print("[Gemini 최종 분석 결과]")
print(results["status"].value_counts())

success = results[results["status"] == "success"]
failed = results[results["status"] != "success"]

print("\n전체 저장 메뉴:", len(results))
print("성공:", len(success))
print("실패:", len(failed))
print("성공률:", f"{len(success) / 4085 * 100:.1f}%")

print("\n[성공한 메뉴 샘플]")
display(
    success[
        ["menu_name", "gemini_estimated_ingredients"]
    ].head(10)
)

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path.cwd()

files = {
    "메뉴 매칭": BASE / "menu_matching_classified.csv",
    "Qwen 분석": BASE / "local_llm_output_v5_full" / "llm_ingredient_extraction_v5.csv",
    "Gemini 분석": BASE / "gemini_ingredient_results.csv",
    "공공데이터": BASE / "public_ingredients_clean.csv"
}

for name, path in files.items():
    print(f"\n[{name}]")
    print("파일 존재:", path.exists())

    if path.exists():
        df = pd.read_csv(path, low_memory=False)
        print("행 수:", len(df))
        print("컬럼:", df.columns.tolist())

In [ ]:
import pandas as pd
import json
from pathlib import Path

BASE = Path.cwd()

# 1. 파일 불러오기
menu = pd.read_csv(BASE / "menu_matching_classified.csv", dtype=str).fillna("")
qwen = pd.read_csv(
    BASE / "local_llm_output_v5_full" / "llm_ingredient_extraction_v5.csv",
    dtype=str
).fillna("")
gemini = pd.read_csv(BASE / "gemini_ingredient_results.csv", dtype=str).fillna("")
public = pd.read_csv(BASE / "public_ingredients_clean.csv", dtype=str).fillna("")

KEY = "preprocess_row_id"

# 2. ID 중복 확인
for name, df in [("메뉴", menu), ("Qwen", qwen), ("Gemini", gemini)]:
    if df[KEY].duplicated().any():
        raise ValueError(f"{name} 데이터에 중복 ID가 있어요.")

# 3. Qwen 결과 연결
qwen_cols = [
    KEY, "explicit_ingredients", "explicit_components",
    "evidence_level", "information_status", "extraction_status"
]
result = menu.merge(
    qwen[qwen_cols],
    on=KEY,
    how="left",
    validate="one_to_one"
)

# 4. Gemini 결과 연결
gemini_cols = [
    KEY, "gemini_estimated_ingredients", "status"
]
gemini_data = gemini[gemini_cols].rename(
    columns={"status": "gemini_status"}
)

result = result.merge(
    gemini_data,
    on=KEY,
    how="left",
    validate="one_to_one"
)

# 5. 공공데이터 알레르기 참고 정보 연결
# 동일 fd_Code가 여러 행일 수 있으므로 먼저 집계
def unique_join(series):
    values = [str(v).strip() for v in series if str(v).strip()]
    return " | ".join(dict.fromkeys(values))

public_ref = (
    public.groupby("fd_Code", as_index=False)
    .agg(
        public_reference_food_name=("food_Nm", unique_join),
        public_reference_allergens=("allergens_normalized", unique_join),
        public_reference_allergy_raw=("allrgy_Info", unique_join)
    )
)

if "top1_fd_Code" not in result.columns:
    raise ValueError("메뉴 매칭 CSV에 top1_fd_Code 컬럼이 없습니다.")

result = result.merge(
    public_ref,
    left_on="top1_fd_Code",
    right_on="fd_Code",
    how="left",
    validate="many_to_one"
)

# 6. 정보 출처 및 검증 여부 구분
result["ingredient_source"] = "NONE"

has_qwen = result["explicit_ingredients"].fillna("").apply(
    lambda x: x.strip() not in ("", "[]")
)

has_gemini = (
    result["gemini_status"].eq("success")
    & result["gemini_estimated_ingredients"].fillna("").apply(
        lambda x: x.strip() not in ("", "[]")
    )
)

result.loc[has_qwen, "ingredient_source"] = "QWEN_TEXT_EXTRACTION"
result.loc[has_gemini & ~has_qwen, "ingredient_source"] = "GEMINI_ESTIMATED"
result.loc[has_gemini & has_qwen, "ingredient_source"] = "QWEN_AND_GEMINI"

result["restaurant_ingredient_verification"] = "NOT_VERIFIED"
result["allergy_safety_status"] = "UNKNOWN_NOT_VERIFIED"
result["public_reference_status"] = "UNVERIFIED_SIMILAR_FOOD"

# 7. 최종 검증 및 저장
assert len(result) == len(menu) == 6034
assert result[KEY].is_unique

output = BASE / "menugen_backend_integrated.csv"
result.to_csv(output, index=False, encoding="utf-8-sig")

print("통합 완료!")
print("전체 메뉴:", len(result))
print("Qwen 재료 정보:", has_qwen.sum())
print("Gemini 재료 정보:", has_gemini.sum())
print("정보 출처별 메뉴 수:")
print(result["ingredient_source"].value_counts())
print("저장 위치:", output)

In [ ]:
import pandas as pd
import json

df = pd.read_csv("menugen_backend_integrated.csv", dtype=str).fillna("")

print("=== 최종 CSV 검증 ===")

# 1. 전체 메뉴 수
print("전체 메뉴:", len(df))

# 2. 메뉴 ID 중복
print("중복 ID:", df["preprocess_row_id"].duplicated().sum())

# 3. 필수 정보 누락
print("메뉴명 누락:", df["menu_name"].eq("").sum())
print("식당명 누락:", df["restaurant_name"].eq("").sum())

# 4. Gemini 성공했지만 재료가 없는 경우
empty_gemini = df[
    (df["gemini_status"] == "success") &
    (df["gemini_estimated_ingredients"].isin(["", "[]"]))
]
print("Gemini 성공/재료 없음:", len(empty_gemini))

# 5. 공공데이터 연결 여부
print("공공데이터 참고 정보 연결:", df["public_reference_food_name"].ne("").sum())

# 6. 정보 출처 확인
print("\n재료 정보 출처:")
print(df["ingredient_source"].value_counts())

# 7. 알레르기 안전성 표시 확인
print("\n알레르기 안전성 상태:")
print(df["allergy_safety_status"].value_counts())

# 8. 샘플 확인
display(df[[
    "restaurant_name",
    "menu_name",
    "explicit_ingredients",
    "gemini_estimated_ingredients",
    "public_reference_allergens",
    "ingredient_source"
]].head(10))

In [ ]:
import pandas as pd

df = pd.read_csv(
    "menugen_backend_integrated.csv",
    dtype=str
).fillna("")

backend_cols = [
    "preprocess_row_id",
    "place_id",
    "restaurant_name",
    "menu_name",
    "description",
    "explicit_ingredients",
    "explicit_components",
    "gemini_estimated_ingredients",
    "ingredient_source",
    "evidence_level",
    "information_status",
    "gemini_status",
    "top1_fd_Code",
    "public_reference_food_name",
    "public_reference_allergens",
    "public_reference_allergy_raw",
    "public_reference_status",
    "restaurant_ingredient_verification",
    "allergy_safety_status"
]

missing = [c for c in backend_cols if c not in df.columns]
if missing:
    raise ValueError(f"누락된 컬럼: {missing}")

backend = df[backend_cols].copy()

backend.to_csv(
    "menugen_backend_final.csv",
    index=False,
    encoding="utf-8-sig"
)

print("백엔드 전달용 CSV 생성 완료!")
print("전체 메뉴:", len(backend))
print("전체 컬럼:", len(backend.columns))
print("중복 ID:", backend["preprocess_row_id"].duplicated().sum())
print("파일명: menugen_backend_final.csv")